In [ ]:
import os
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

# =========================
# PATH CONFIG (KAGGLE)
# =========================
DATA_PATH = "data/processed/"
SAVE_PATH = "/kaggle/working/processed"   # writable in Kaggle
MODEL_PATH = "/kaggle/working/models"

os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

# =========================
# LOAD DATA
# =========================
print(os.listdir(DATA_PATH))

X_train = pd.read_csv(f"{DATA_PATH}/X_train.csv")
X_test  = pd.read_csv(f"{DATA_PATH}/X_test.csv")

y_train = pd.read_csv(f"{DATA_PATH}/y_train.csv")
y_test  = pd.read_csv(f"{DATA_PATH}/y_test.csv")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
print("Setup done!")

### Load Data 

In [ ]:
X_train = pd.read_csv(f"{DATA_PATH}/X_train.csv")
X_val   = pd.read_csv(f"{DATA_PATH}/X_val.csv")
X_test  = pd.read_csv(f"{DATA_PATH}/X_test.csv")

y_train = pd.read_csv(f"{DATA_PATH}/y_train.csv").values.ravel()
y_val   = pd.read_csv(f"{DATA_PATH}/y_val.csv").values.ravel()
y_test  = pd.read_csv(f"{DATA_PATH}/y_test.csv").values.ravel()

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Features: {X_train.columns.tolist()}")
print(f"Class distribution (train): {pd.Series(y_train).value_counts().to_dict()}")

In [ ]:
# Stream 1 — Clinical features
clinical_features = ["Age", "Gender"]
clinical_features = [c for c in clinical_features if c in X_train.columns]

# Stream 2 — Lab features
lab_features = [c for c in X_train.columns
    if c not in clinical_features]

# Stream 3 — All features (biospecimen approximation)
bio_features = X_train.columns.tolist()

print(f"Stream 1 (Clinical): {clinical_features}")
print(f"Stream 2 (Lab): {lab_features}")
print(f"Stream 3 (Bio): {len(bio_features)} features")

In [ ]:
# Convert to PyTorch tensors
def to_tensor(df, cols=None):
    if cols:
        data = df[cols].values.astype(np.float32)
    else:
        data = df.values.astype(np.float32)
    return torch.tensor(data)

# Stream tensors
X_train_clin = to_tensor(X_train, clinical_features) if clinical_features else to_tensor(X_train)
X_train_lab= to_tensor(X_train, lab_features)
X_train_bio = to_tensor(X_train)

X_val_clin= to_tensor(X_val, clinical_features) if clinical_features else to_tensor(X_val)
X_val_lab= to_tensor(X_val, lab_features)
X_val_bio= to_tensor(X_val)

X_test_clin  = to_tensor(X_test, clinical_features) if clinical_features else to_tensor(X_test)
X_test_lab= to_tensor(X_test, lab_features)
X_test_bio= to_tensor(X_test)

y_train_t = torch.tensor(y_train.astype(np.float32))
y_val_t= torch.tensor(y_val.astype(np.float32))
y_test_t= torch.tensor(y_test.astype(np.float32))

print(f"Clinical tensor : {X_train_clin.shape}")
print(f"Lab tensor: {X_train_lab.shape}")
print(f"Bio tensor: {X_train_bio.shape}")
print("Tensors ready!")